# TransformerLens to TorchLens Migration

Self-contained: a seeded tiny PyTorch model, no downloads, CPU only. The shape of the workflow is what carries over to a real model.


TransformerLens users often start from `run_with_cache`, named hook points, and activation
patching. TorchLens runs the model as regular PyTorch and records every operation, so the
migration is mostly replacing cache keys with selector queries and hook-point names with
resolved TorchLens sites. The model below is a tiny stand-in; for a real `HookedTransformer`,
use its tokenizer and resolve the activation function and module path that model actually uses.
The full guide is [docs/migration/from_transformerlens.md](../../docs/migration/from_transformerlens.md).


In [ ]:
import torch
from torch import nn

import torchlens as tl

SEED = 1603
torch.manual_seed(SEED)
torch.set_grad_enabled(False)


class MockHookedTransformer(nn.Module):
    """Tiny local stand-in for a TransformerLens HookedTransformer."""

    def __init__(self, vocab_size: int = 12, width: int = 10) -> None:
        """Initialize a minimal token model."""
        super().__init__()
        self.embed = nn.Embedding(vocab_size, width)
        self.blocks = nn.ModuleList([nn.Sequential(nn.Linear(width, width), nn.ReLU())])
        self.unembed = nn.Linear(width, vocab_size, bias=False)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        """Return token logits."""
        x = self.embed(tokens)
        for block in self.blocks:
            x = x + block(x)
        return self.unembed(x)

In [ ]:
model = MockHookedTransformer().eval()
tokens = torch.tensor([[1, 4, 5, 2]])
trace = tl.trace(model, tokens)
print(f"TorchLens recorded {len(trace.ops)} operations")
print([site.layer_label for site in trace.find_sites(tl.func("relu"))])

Migration pattern: `cache['blocks.0.mlp.hook_post']` becomes a selector query such as `tl.func('relu') & tl.in_module('blocks.0')`, then `find_sites(...).first().out` returns the tensor.


In [ ]:
mlp_post = trace.find_sites(tl.func("relu") & tl.in_module("blocks.0")).first()
print(mlp_post.layer_label)
print(tuple(mlp_post.out.shape))

For intervention-style code, `run_with_hooks` maps to a fork plus `do(site, hook)`. Plain `do` replays the recorded graph, the fastest option when that graph is stable; `do(site, hook, model=model, inputs=x)` reruns the original model and is the closer analogue when it should execute again.


In [ ]:
ablated = trace.fork("zero_mlp_post")
ablated.do(tl.label(mlp_post.layer_label), tl.zero_ablate())

clean_out = trace.output_ops[-1].out
ablated_out = ablated.output_ops[-1].out
print(f"output L2 change: {torch.linalg.vector_norm(ablated_out - clean_out):.4f}")